In [1]:
import pandas as pd
import sys
sys.path.insert(0, "../../run")
from const import REPO_PATH
from inference_config import INFERENCE_CONFIG

sys.path.insert(1, f"{REPO_PATH}")
from src.feature.feature_encoders import TeamEncoder, TeamLagFeatureGenerator, PreviousSeasonTeamAverager, TeamRestDaysCalculator

In [3]:
seasons=sorted(INFERENCE_CONFIG['seasons'])
data_dfs=[pd.read_csv(f"{INFERENCE_CONFIG['processed_path']}/{season}/all_data_df.csv") for season in seasons]

## Generate feature vector for a single match using all feature classes

This function takes a list of season DataFrames, a home/away team, and a date, and generates the feature vector using the fitted encoder and all feature classes in feature_encoders.py. It uses the existing encoder and does not retrain any feature class.

In [4]:
import pandas as pd
from src.feature.feature_encoders import TeamEncoder, TeamLagFeatureGenerator, PreviousSeasonTeamAverager, TeamRestDaysCalculator

def generate_features_for_match(
    season_dfs,
    home_team,
    away_team,
    match_date,
    encoder_path,
    lookback=5,
    decay_factor=1.0,
    home_col='home',
    away_col='away',
    date_col='date'
):
    """
    Generate a feature vector for a single match using all feature classes.

    Args:
        season_dfs (list of pd.DataFrame): List of season DataFrames (chronological order).
        home_team (str): Home team name.
        away_team (str): Away team name.
        match_date (str or pd.Timestamp): Date of the match.
        encoder_path (str): Path to the fitted TeamEncoder.
        lookback (int): Lookback for lag features.
        decay_factor (float): Decay for previous season features.
        home_col, away_col, date_col: Column names.

    Returns:
        pd.DataFrame: Single-row DataFrame with all features for the match.
    """
    match_date = pd.to_datetime(match_date)

    # --- Team Encoding ---
    encoder = TeamEncoder.load(encoder_path)
    team_encoding = encoder.encoder.transform([[home_team, away_team]])
    team_encoding_cols = encoder.encoder.get_feature_names_out(['encoded_home', 'encoded_away'])
    team_encoding_df = pd.DataFrame(team_encoding, columns=team_encoding_cols)
    team_encoding_df[home_col] = home_team
    team_encoding_df[away_col] = away_team
    team_encoding_df[date_col] = match_date

    # --- Lag Features ---
    lag_generator = TeamLagFeatureGenerator(lookback=lookback, date_col=date_col, home_col=home_col, away_col=away_col)
    lag_features_df = lag_generator.transform(season_dfs)
    lag_row = lag_features_df[
        (lag_features_df[home_col] == home_team) &
        (lag_features_df[away_col] == away_team) &
        (lag_features_df[date_col] <= match_date)
    ].sort_values(date_col, ascending=False).head(1)
    if lag_row.empty:
        lag_row = pd.DataFrame(columns=lag_features_df.columns)
    lag_row = lag_row.copy()
    lag_row[home_col] = home_team
    lag_row[away_col] = away_team
    lag_row[date_col] = match_date

    # --- Previous Season Features ---
    prev_season_generator = PreviousSeasonTeamAverager(decay_factor=decay_factor, date_col=date_col, home_col=home_col, away_col=away_col)
    prev_season_features_df = prev_season_generator.transform(season_dfs)
    prev_row = prev_season_features_df[
        (prev_season_features_df[home_col] == home_team) &
        (prev_season_features_df[away_col] == away_team) &
        (prev_season_features_df[date_col] <= match_date)
    ].sort_values(date_col, ascending=False).head(1)
    if prev_row.empty:
        prev_row = pd.DataFrame(columns=prev_season_features_df.columns)
    prev_row = prev_row.copy()
    prev_row[home_col] = home_team
    prev_row[away_col] = away_team
    prev_row[date_col] = match_date

    # --- Rest Days Features ---
    rest_days_generator = TeamRestDaysCalculator(home_col=home_col, away_col=away_col, date_col=date_col)
    rest_days_features_df = rest_days_generator.transform(season_dfs)
    rest_row = rest_days_features_df[
        (rest_days_features_df[home_col] == home_team) &
        (rest_days_features_df[away_col] == away_team) &
        (rest_days_features_df[date_col] <= match_date)
    ].sort_values(date_col, ascending=False).head(1)
    if rest_row.empty:
        rest_row = pd.DataFrame(columns=rest_days_features_df.columns)
    rest_row = rest_row.copy()
    rest_row[home_col] = home_team
    rest_row[away_col] = away_team
    rest_row[date_col] = match_date

    # --- Merge all features ---
    merged = team_encoding_df.merge(
        lag_row, on=[home_col, away_col, date_col], how='left', suffixes=('', '_lag')
    ).merge(
        prev_row, on=[home_col, away_col, date_col], how='left', suffixes=('', '_prev')
    ).merge(
        rest_row, on=[home_col, away_col, date_col], how='left', suffixes=('', '_rest')
    )

    merged = merged.fillna(-1)
    return merged

In [7]:
feature_vector=generate_features_for_match(season_dfs=data_dfs,
                            home_team='Chelsea',
                            away_team='Bournemouth',
                            match_date='2025-01-14',
                            encoder_path=f"{INFERENCE_CONFIG['data_processor_path']}/team_encoder.pkl")

/Users/tianqihuang/anaconda3/envs/betbot/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [ ]:
feature_vector['encoded_home_Chelsea'], feature_vector['encoded_away_Bournemouth'], 

(0    1.0
 Name: encoded_home_Chelsea, dtype: float64,
 0    1.0
 Name: encoded_away_Bournemouth, dtype: float64)